[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/07_trees.ipynb)

*DSA Interview Handbook · Notebook 7 of 14*

# Trees & BST

Recursion on left and right subtrees, BFS by level, and the ordering rule of binary search trees.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Trust the recursion: assume your function works on the children, then just combine their answers.

**What interviewers really test here:** DFS vs BFS, BST properties, and what each call returns.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Tree vocabulary *(Basics)*
- [ ] 2. Recursion on trees *(Basics)*
- [ ] 3. Breadth-first search: level by level *(Core)*
- [ ] 4. Bottom-up: return a value, update a global *(Core)*
- [ ] 5. Top-down: pass information down *(Core)*
- [ ] 6. Binary search trees *(Core)*
- [ ] 7. Building, serialising and ancestors *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **tree** is a **non-linear, hierarchical** data structure made of nodes connected by edges, with one special node called the **root** and **no cycles**. Every node except the root has exactly one parent.



A **binary tree** is a tree in which each node has at most two children, called the **left child** and the **right child**.



A **binary search tree (BST)** is a binary tree where, for every node, all keys in its left subtree are smaller and all keys in its right subtree are larger than the node's key.

### Terminology

| Term | Meaning |
| --- | --- |
| Root | top node, no parent |
| Parent / child | a node directly above / below another |
| Siblings | nodes with the same parent |
| Leaf (external node) | node with no children |
| Internal node | node with at least one child |
| Edge | connection between parent and child |
| Path | sequence of nodes connected by edges |
| Depth of a node | number of edges from the root to it (root has depth 0) |
| Height of a node | edges on the longest path from it down to a leaf (leaf has height 0) |
| Height of the tree | height of the root |
| Level | all nodes at the same depth |
| Degree | number of children of a node |
| Subtree | a node and all its descendants |
| Ancestor / descendant | nodes on the path above / below |

### Types of binary trees

| Type | Definition |
| --- | --- |
| **Full (strict)** | every node has 0 or 2 children |
| **Complete** | all levels full except possibly the last, which is filled from the left (heaps are complete) |
| **Perfect** | all internal nodes have 2 children and all leaves are at the same level |
| **Balanced** | for every node, left and right subtree heights differ by at most 1 (height O(log n)) |
| **Degenerate (skewed)** | every node has one child — effectively a linked list (height n − 1) |



#### Self-balancing BSTs



Plain BSTs can degenerate to O(n). **AVL trees** (strict balance by rotations) and **Red–Black trees** (looser colour rules, used in Java's `TreeMap` and C++ `std::map`) keep height O(log n) after every insert and delete. Also: B-trees / B+ trees for databases and file systems, tries for strings, segment trees for range queries.

### Important properties

- A tree with **n** nodes has exactly **n − 1** edges.
- Maximum nodes at level i (root at level 0): **2ⁱ**.
- Maximum nodes in a binary tree of height h: **2ʰ⁺¹ − 1** (a perfect tree).
- Minimum height of a binary tree with n nodes: **⌈log₂(n + 1)⌉ − 1**.
- In a full binary tree, leaves = internal nodes + 1.
- In a complete binary tree stored in an array, the children of index i are **2i + 1** and **2i + 2**, and the parent is **(i − 1) // 2**.
- Number of structurally different binary trees with n nodes is the Catalan number C(n) = (2n)! / ((n + 1)! n!).

### Traversals

| Traversal | Order | For tree [1, 2, 3, 4, 5] | Uses |
| --- | --- | --- | --- |
| **Preorder** (DFS) | Root, Left, Right | 1 2 4 5 3 | copy / serialise a tree, prefix expressions |
| **Inorder** (DFS) | Left, Root, Right | 4 2 5 1 3 | sorted order of a BST, infix expressions |
| **Postorder** (DFS) | Left, Right, Root | 4 5 2 3 1 | delete a tree, compute heights/sizes, postfix expressions |
| **Level order** (BFS) | level by level, left to right | 1 2 3 4 5 | shortest depth, per-level problems |



All traversals are O(n) time. DFS uses O(h) space (recursion stack); BFS uses O(w) space (widest level). **Morris traversal** does inorder in O(1) space by temporarily rewiring right pointers.



A tree can be uniquely rebuilt from **inorder + preorder** or **inorder + postorder** (with distinct values), but not from preorder + postorder alone.

### BST operations and complexity

| Operation | Balanced BST | Skewed BST | How |
| --- | --- | --- | --- |
| Search | O(log n) | O(n) | go left if smaller, right if larger |
| Insert | O(log n) | O(n) | search for the spot, attach a new leaf |
| Delete | O(log n) | O(n) | leaf: remove; one child: link child to parent; two children: replace with inorder successor (smallest in right subtree), then delete that |
| Min / max | O(log n) | O(n) | leftmost / rightmost node |
| Inorder (sorted output) | O(n) | O(n) | left, node, right |

### Advantages and applications

**👍 Pros**

##### Advantages

- Represents hierarchies naturally
- Balanced BSTs: O(log n) search, insert, delete *and* sorted order
- Range queries, floor/ceiling, k-th smallest



**👎 Cons**

##### Disadvantages

- Unbalanced BSTs degrade to O(n)
- Balancing logic (AVL, Red–Black) is complex
- More memory than arrays (two pointers per node)





#### Applications




- File systems and folder hierarchies; the HTML DOM; organisation charts.
- Database indexes (B-trees, B+ trees).
- Sorted maps and sets (`TreeMap`, `std::map`).
- Expression trees and syntax trees in compilers.
- Decision trees in machine learning; Huffman coding trees in compression.
- Heaps (priority queues) and tries (autocomplete).

---
# Part 2: Lessons, step by step

## Lesson 1: Tree vocabulary

*Level: Basics*

A **tree** is a set of nodes where each node has one parent (except the **root**) and any number of children. In a **binary tree** each node has at most two: `left` and `right`.



```text
        1          ← root (depth 0)
       / \
      2   3        ← depth 1
     / \   \
    4   5   6      ← leaves (no children), depth 2       height = 3 levels
```




- **Leaf**: a node with no children.
- **Depth** of a node: edges from the root to it. **Height** of a tree: number of levels (or edges on the longest root-to-leaf path — check which one the problem means).
- **Subtree**: a node together with everything below it. Every subtree is itself a tree — this is why recursion fits trees so well.
- **Balanced**: height is about log₂ n. A **skewed** tree is basically a linked list with height n.



LeetCode writes trees as level-order lists with `None` for gaps: `[1, 2, 3, 4, 5, None, 6]`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def build_tree_lesson():
    vals = [1, 2, 3, 4, 5, None, 6]
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root


build_tree_lesson()

TreeNode[1, 2, 3, 4, 5, None, 6]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Root from vals[0]
2. vals[1] = 2 becomes the left child of 1
3. vals[2] = 3 becomes the right child of 1
4. vals[3] = 4 becomes the left child of 2
5. vals[4] = 5 becomes the right child of 2
6. vals[6] = 6 becomes the right child of 3

</details>


## Lesson 2: Recursion on trees

*Level: Basics*

Nearly every tree solution has the same shape:



```python
def solve(node):
    if node is None:          # 1. base case: empty tree
        return BASE
    left = solve(node.left)   # 2. trust recursion for the children
    right = solve(node.right)
    return combine(node.val, left, right)   # 3. combine
```



Example — maximum depth: base is 0, combine is `1 + max(left, right)`. Count nodes: `1 + left + right`. Sum: `node.val + left + right`.



#### The three DFS orders



"Visiting" a node means doing your work there. Doing it before the children is **preorder**, between them **inorder**, after them **postorder**. Each tree node is visited once → O(n) time, O(h) stack space.



**💡 Tip:** Don't trace the whole recursion in your head. Assume the calls on the children return correct answers and only check the combine step and the base case.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q3](#q3), [Q4](#q4), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 3: Breadth-first search: level by level

*Level: Core*

DFS dives deep first. **BFS** visits all nodes at depth 0, then depth 1, then depth 2 … using a **queue**.



```python
from collections import deque
q = deque([root])
while q:
    level = []
    for _ in range(len(q)):     # exactly the nodes of this level
        node = q.popleft()
        level.append(node.val)
        if node.left:  q.append(node.left)
        if node.right: q.append(node.right)
    print(level)
```



The `for _ in range(len(q))` trick separates levels. Use BFS for anything "per level": level order, right side view (last of each level), zigzag, minimum depth (first leaf found).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def bfs_tree_lesson():
    from collections import deque
    root = mk_tree([3, 9, 20, None, None, 15, 7])
    q, levels = deque([root]), []
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        levels.append(level)
    return levels


bfs_tree_lesson()

[[3], [9, 20], [15, 7]]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Level 0: [3]; the queue now holds the next level
2. Level 1: [9, 20]; the queue now holds the next level
3. Level 2: [15, 7]; the queue now holds the next level

</details>


**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q14](#q14), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 4: Bottom-up: return a value, update a global

*Level: Core*

Some answers don't pass through the root. **Diameter**: the longest path may go through any node. Solve with a postorder DFS that **returns** one thing (height) and **updates** another (best answer):



```python
best = 0
def height(node):
    nonlocal best
    if not node: return 0
    l, r = height(node.left), height(node.right)
    best = max(best, l + r)        # path that bends at this node
    return 1 + max(l, r)           # what the parent can use
```



The same split solves **balanced tree** (return height or −1) and **max path sum** (return the best one-sided chain, record the best two-sided path).

**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q7](#q7), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 5: Top-down: pass information down

*Level: Core*

Sometimes a node needs to know about its **ancestors**. Pass that info as a parameter:




- **Good nodes**: pass the largest value seen on the path from the root.
- **Validate BST**: pass the allowed range `(low, high)`; going left sets `high = node.val`, going right sets `low = node.val`.
- **Path sums**: pass the remaining target.



Bottom-up = "children tell the parent". Top-down = "parent tells the children". Some problems use both.

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 6: Binary search trees

*Level: Core*

A **BST** keeps one rule at every node: all values in the left subtree are smaller, all in the right subtree are larger.




- **Search / insert**: compare with the node and go left or right — O(h), which is O(log n) when balanced.
- **Inorder traversal gives sorted order.** So "k-th smallest" = the k-th inorder visit; "validate" = inorder values strictly increasing.
- **LCA in a BST**: walk from the root; if both targets are smaller go left, if both larger go right, otherwise you're at the split point — the answer.



**⚠️ Watch out:** Checking only `node.left.val < node.val < node.right.val` is not enough — a value deep in the left subtree can still be too big. Use ranges.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def bst_lesson():
    root = None
    for v in [8, 3, 10, 1, 6, 14, 4]:
        if root is None:
            root = TreeNode(v)
        else:
            node = root
            while True:
                side = "left" if v < node.val else "right"
                nxt = getattr(node, side)
                if nxt is None:
                    setattr(node, side, TreeNode(v))
                    break
                node = nxt
    node, path = root, []
    while node and node.val != 4:
        path.append(node)
        node = node.left if 4 < node.val else node.right
    return root


bst_lesson()

TreeNode[8, 3, 10, 1, 6, None, 14, None, None, 4]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Insert 8: go left if smaller, right if larger
2. Insert 3: go left if smaller, right if larger
3. Insert 10: go left if smaller, right if larger
4. Insert 1: go left if smaller, right if larger
5. Insert 6: go left if smaller, right if larger
6. Insert 14: go left if smaller, right if larger
7. Insert 4: go left if smaller, right if larger
8. Search 4: only one path is checked — O(height)

</details>


**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q13](#q13), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 7: Building, serialising and ancestors

*Level: Advanced*

- **Build from preorder + inorder**: preorder's first value is the root; its position in inorder splits left and right subtrees. A dict of inorder positions makes it O(n).
- **Serialise**: preorder with a marker (`#`) for every missing child is enough to rebuild the exact tree.
- **LCA in a general tree**: each call returns p or q if found below. The first node that gets an answer from both sides is the LCA.
- **Flatten / rewire in place**: find the rightmost node of the left subtree, hang the right subtree there, move left to right (Morris-style, O(1) space).
- **Subtree check**: serialise both trees and use substring search.



**💡 Tip:** Python's recursion limit is about 1000. For trees that could be 10⁵ deep, write the DFS with an explicit stack.

**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q20](#q20), [Q11](#q11), [Q18](#q18), [Q8](#q8). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [5]:
T1 = lambda a: (mk_tree(a[0]),)

def _lca_prep(a):
    root = mk_tree(a[0])
    return root, find(root, a[1]), find(root, a[2])


<a id="q1"></a>
### Q1. Binary Tree Inorder Traversal

🟢 Warm-up · *Iterative stack*

Return the inorder traversal (left, node, right) of a binary tree.

**Examples**

- `root = [1, None, 2, 3]` → `[1, 3, 2]`
- `root = [1, 2, 3, 4, 5, None, 8, None, None, 6, 7, 9]` → `[4, 2, 6, 5, 7, 1, 3, 9, 8]`
- `root = []` → `[]`

In [ ]:
def inorder_traversal(root):
    """
    Return the inorder traversal (left, node, right) of a binary tree.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(inorder_traversal, [(([1, None, 2, 3],), [1, 3, 2]), (([1, 2, 3, 4, 5, None, 8, None, None, 6, 7, 9],), [4, 2, 6, 5, 7, 1, 3, 9, 8]), (([],), [])], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive DFS: visit left, record the node, visit right.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same order with an explicit stack: push the whole left spine, pop and record, then move to the right child. (Morris traversal reaches O(1) space.)

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recursive DFS: visit left, record the node, visit right.

```python
def inorder_traversal(root):
    res = []

    def dfs(node):
        if node:
            dfs(node.left)
            res.append(node.val)
            dfs(node.right)
    dfs(root)
    return res
```

**Cost:** O(n) time, O(h) call stack space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Same order with an explicit stack: push the whole left spine, pop and record, then move to the right child. (Morris traversal reaches O(1) space.)

```python
def inorder_traversal(root):
    res, stack, cur = [], [], root
    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        res.append(cur.val)
        cur = cur.right
    return res
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive DFS: visit left, record the node, visit right. That costs O(n) time and O(h) call stack space."

"I can do better: Same order with an explicit stack: push the whole left spine, pop and record, then move to the right child. (Morris traversal reaches O(1) space.) That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [4, 2, 6, 1, 3, 5]`

1. Pop 1 and record it, then go right
2. Pop 2 and record it, then go right
3. Pop 3 and record it, then go right
4. Pop 4 and record it, then go right
5. Pop 5 and record it, then go right
6. Pop 6 and record it, then go right

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/binary-tree-inorder-traversal)

<a id="q2"></a>
### Q2. Maximum Depth of Binary Tree

🟢 Warm-up · *Postorder height*

Return the number of nodes on the longest root-to-leaf path.

**Examples**

- `root = [3, 9, 20, None, None, 15, 7]` → `3`
- `root = [1, None, 2]` → `2`
- `root = []` → `0`

In [ ]:
def max_depth(root):
    """
    Return the number of nodes on the longest root-to-leaf path.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_depth, [(([3, 9, 20, None, None, 15, 7],), 3), (([1, None, 2],), 2), (([],), 0)], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

List every root-to-leaf path and take the longest.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

depth(node) = 1 + max(depth(left), depth(right)); an empty tree has depth 0.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** List every root-to-leaf path and take the longest.

```python
def max_depth(root):
    paths = []

    def walk(node, path):
        if not node:
            return
        path = path + [node.val]
        if not node.left and not node.right:
            paths.append(path)
        walk(node.left, path)
        walk(node.right, path)
    walk(root, [])
    return max((len(p) for p in paths), default=0)
```

**Cost:** O(n · h) time, O(n · h) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** depth(node) = 1 + max(depth(left), depth(right)); an empty tree has depth 0.

```python
def max_depth(root):
    def depth(node):
        if not node:
            return 0
        d = 1 + max(depth(node.left), depth(node.right))
        return d
    return depth(root)
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: List every root-to-leaf path and take the longest. That costs O(n · h) time and O(n · h) space."

"I can do better: depth(node) = 1 + max(depth(left), depth(right)); an empty tree has depth 0. That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [3, 9, 20, None, None, 15, 7]`

1. depth(9) = 1
2. depth(15) = 1
3. depth(7) = 1
4. depth(20) = 2
5. depth(3) = 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/maximum-depth-of-binary-tree)

<a id="q3"></a>
### Q3. Invert Binary Tree

🟢 Warm-up · *Swap children recursively*

Mirror a binary tree: swap every node's left and right children.

**Examples**

- `root = [4, 2, 7, 1, 3, 6, 9]` → `[4, 7, 2, 9, 6, 3, 1]`
- `root = [2, 1, 3]` → `[2, 3, 1]`
- `root = []` → `[]`

In [ ]:
def invert_tree(root):
    """
    Mirror a binary tree: swap every node's left and right children.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(invert_tree, [(([4, 2, 7, 1, 3, 6, 9],), [4, 7, 2, 9, 6, 3, 1]), (([2, 1, 3],), [2, 3, 1]), (([],), [])], prep=T1, post=tree_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a brand-new mirrored copy of the tree.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

At every node swap left and right, then invert both subtrees in place.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Build a brand-new mirrored copy of the tree.

```python
def invert_tree(root):
    if not root:
        return None
    return TreeNode(root.val, invert_tree(root.right), invert_tree(root.left))
```

**Cost:** O(n) time, O(n) new nodes space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** At every node swap left and right, then invert both subtrees in place.

```python
def invert_tree(root):
    top = root

    def inv(node):
        if not node:
            return
        node.left, node.right = node.right, node.left
        inv(node.left)
        inv(node.right)
    inv(root)
    return root
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a brand-new mirrored copy of the tree. That costs O(n) time and O(n) new nodes space."

"I can do better: At every node swap left and right, then invert both subtrees in place. That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [4, 2, 7, 1, 3, 6, 9]`

1. Swap the children of 4
2. Swap the children of 7
3. Swap the children of 9
4. Swap the children of 6
5. Swap the children of 2
6. Swap the children of 3
7. Swap the children of 1

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/invert-binary-tree)

<a id="q4"></a>
### Q4. Same Tree

🟢 Warm-up · *Parallel recursion*

Return `True` if two binary trees have the same shape and values.

**Examples**

- `p = [1, 2, 3], q = [1, 2, 3]` → `True`
- `p = [1, 2], q = [1, None, 2]` → `False`
- `p = [1, 2, 1], q = [1, 1, 2]` → `False`

In [ ]:
def is_same_tree(p, q):
    """
    Return True if two binary trees have the same shape and values.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_same_tree, [(([1, 2, 3], [1, 2, 3]), True), (([1, 2], [1, None, 2]), False), (([1, 2, 1], [1, 1, 2]), False)], prep=lambda a: (mk_tree(a[0]), mk_tree(a[1])))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Serialise both trees with None markers and compare the strings.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Recurse on both trees together: both empty → True; one empty or values differ → False; else compare left with left and right with right.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Serialise both trees with None markers and compare the strings.

```python
def is_same_tree(p, q):
    def ser(n):
        return "#" if not n else f"({n.val},{ser(n.left)},{ser(n.right)})"
    return ser(p) == ser(q)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Recurse on both trees together: both empty → True; one empty or values differ → False; else compare left with left and right with right.

```python
def is_same_tree(p, q):
    rp, rq = p, q

    def same(a, b):
        if not a and not b:
            return True
        if not a or not b or a.val != b.val:
            return False
        return same(a.left, b.left) and same(a.right, b.right)
    return same(p, q)
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Serialise both trees with None markers and compare the strings. That costs O(n) time and O(n) space."

"I can do better: Recurse on both trees together: both empty → True; one empty or values differ → False; else compare left with left and right with right. That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `p = [1, 2, 3, 4], q = [1, 2, 3, 5]`

1. Both 1 — compare children
2. Both 2 — compare children
3. Mismatch: 4 vs 5

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/same-tree)

<a id="q5"></a>
### Q5. Symmetric Tree

🟢 Warm-up · *Mirror recursion*

Return `True` if the tree is a mirror of itself around its centre.

**Examples**

- `root = [1, 2, 2, 3, 4, 4, 3]` → `True`
- `root = [1, 2, 2, None, 3, None, 3]` → `False`

In [ ]:
def is_symmetric(root):
    """
    Return True if the tree is a mirror of itself around its centre.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_symmetric, [(([1, 2, 2, 3, 4, 4, 3],), True), (([1, 2, 2, None, 3, None, 3],), False)], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Make an inverted copy and check it equals the original.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compare the left subtree with the right subtree mirrored: outer pair (a.left, b.right) and inner pair (a.right, b.left).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Make an inverted copy and check it equals the original.

```python
def is_symmetric(root):
    def mirror(n):
        return None if not n else TreeNode(n.val, mirror(n.right), mirror(n.left))

    def same(a, b):
        if not a or not b:
            return a is b
        return a.val == b.val and same(a.left, b.left) and same(a.right, b.right)
    return same(root, mirror(root))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Compare the left subtree with the right subtree mirrored: outer pair (a.left, b.right) and inner pair (a.right, b.left).

```python
def is_symmetric(root):
    def mirror(a, b):
        if not a or not b:
            return a is b
        return a.val == b.val and mirror(a.left, b.right) and mirror(a.right, b.left)
    return mirror(root.left, root.right)
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Make an inverted copy and check it equals the original. That costs O(n) time and O(n) space."

"I can do better: Compare the left subtree with the right subtree mirrored: outer pair (a.left, b.right) and inner pair (a.right, b.left). That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [1, 2, 2, 3, 4, 4, 3]`

1. Compare 2 with its mirror 2
2. Compare 3 with its mirror 3
3. Compare 4 with its mirror 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/symmetric-tree)

<a id="q6"></a>
### Q6. Diameter of Binary Tree

🟢 Warm-up · *Height + global best*

Return the number of edges on the longest path between any two nodes.

**Examples**

- `root = [1, 2, 3, 4, 5]` → `3`
- `root = [1, 2]` → `1`

In [ ]:
def diameter_of_binary_tree(root):
    """
    Return the number of edges on the longest path between any two nodes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(diameter_of_binary_tree, [(([1, 2, 3, 4, 5],), 3), (([1, 2],), 1)], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every node, compute left and right heights with separate calls.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One postorder pass: each node returns its height, and left + right height through it is a candidate diameter.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every node, compute left and right heights with separate calls.

```python
def diameter_of_binary_tree(root):
    def height(n):
        return 0 if not n else 1 + max(height(n.left), height(n.right))

    def best(n):
        if not n:
            return 0
        return max(height(n.left) + height(n.right), best(n.left), best(n.right))
    return best(root)
```

**Cost:** O(n²) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** One postorder pass: each node returns its height, and left + right height through it is a candidate diameter.

```python
def diameter_of_binary_tree(root):
    best = 0

    def height(node):
        nonlocal best
        if not node:
            return 0
        l, r = height(node.left), height(node.right)
        best = max(best, l + r)
        return 1 + max(l, r)

    height(root)
    return best
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every node, compute left and right heights with separate calls. That costs O(n²) time and O(h) space."

"I can do better: One postorder pass: each node returns its height, and left + right height through it is a candidate diameter. That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [1, 2, 3, 4, 5, None, None, 6, None, None, 7]`

1. At 6: left height 0 + right height 0 = 0
2. At 4: left height 1 + right height 0 = 1
3. At 7: left height 0 + right height 0 = 0
4. At 5: left height 0 + right height 1 = 1
5. At 2: left height 2 + right height 2 = 4
6. At 3: left height 0 + right height 0 = 0
7. At 1: left height 3 + right height 1 = 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/diameter-of-binary-tree)

<a id="q7"></a>
### Q7. Balanced Binary Tree

🟢 Warm-up · *Height or −1*

A tree is height-balanced if every node's subtrees differ in height by at most 1. Check it.

**Examples**

- `root = [3, 9, 20, None, None, 15, 7]` → `True`
- `root = [1, 2, 2, 3, 3, None, None, 4, 4]` → `False`
- `root = []` → `True`

In [ ]:
def is_balanced(root):
    """
    A tree is height-balanced if every node's subtrees differ in height by at
    most 1. Check it.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_balanced, [(([3, 9, 20, None, None, 15, 7],), True), (([1, 2, 2, 3, 3, None, None, 4, 4],), False), (([],), True)], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

At every node compare the heights of both subtrees, computing heights from scratch.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Return the height bottom-up, or −1 as soon as any subtree is unbalanced — the −1 short-circuits upward.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** At every node compare the heights of both subtrees, computing heights from scratch.

```python
def is_balanced(root):
    def height(n):
        return 0 if not n else 1 + max(height(n.left), height(n.right))
    if not root:
        return True
    return abs(height(root.left) - height(root.right)) <= 1 and is_balanced(root.left) and is_balanced(root.right)
```

**Cost:** O(n²) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Return the height bottom-up, or −1 as soon as any subtree is unbalanced — the −1 short-circuits upward.

```python
def is_balanced(root):
    def check(node):
        if not node:
            return 0
        l, r = check(node.left), check(node.right)
        if l < 0 or r < 0 or abs(l - r) > 1:
            return -1
        return 1 + max(l, r)
    return check(root) >= 0
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: At every node compare the heights of both subtrees, computing heights from scratch. That costs O(n²) time and O(h) space."

"I can do better: Return the height bottom-up, or −1 as soon as any subtree is unbalanced — the −1 short-circuits upward. That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [3, 9, 20, None, None, 15, 7]`

1. 9: heights 0 and 0 → height 1
2. 15: heights 0 and 0 → height 1
3. 7: heights 0 and 0 → height 1
4. 20: heights 1 and 1 → height 2
5. 3: heights 1 and 2 → height 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/balanced-binary-tree)

<a id="q8"></a>
### Q8. Subtree of Another Tree

🟢 Warm-up · *Serialise + substring*

Return `True` if `subRoot` appears as a full subtree of `root`.

**Examples**

- `root = [3, 4, 5, 1, 2], sub = [4, 1, 2]` → `True`
- `root = [3, 4, 5, 1, 2, None, None, None, None, 0], sub = [4, 1, 2]` → `False`

In [ ]:
def is_subtree(root, sub):
    """
    Return True if subRoot appears as a full subtree of root.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_subtree, [(([3, 4, 5, 1, 2], [4, 1, 2]), True), (([3, 4, 5, 1, 2, None, None, None, None, 0], [4, 1, 2]), False)], prep=lambda a: (mk_tree(a[0]), mk_tree(a[1])))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every node of root, check whether the tree starting there equals subRoot.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Serialise both trees in preorder with markers for None and delimiters around values; subRoot is a subtree exactly when its string is a substring. (Use KMP for a guaranteed linear search.)

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every node of root, check whether the tree starting there equals subRoot.

```python
def is_subtree(root, sub):
    def same(a, b):
        if not a or not b:
            return a is b
        return a.val == b.val and same(a.left, b.left) and same(a.right, b.right)
    if not root:
        return False
    return same(root, sub) or is_subtree(root.left, sub) or is_subtree(root.right, sub)
```

**Cost:** O(m · n) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Serialise both trees in preorder with markers for None and delimiters around values; subRoot is a subtree exactly when its string is a substring. (Use KMP for a guaranteed linear search.)

```python
def is_subtree(root, sub):
    def ser(node, out):
        if not node:
            out.append("#")
            return
        out.append(f"^{node.val}")
        ser(node.left, out)
        ser(node.right, out)

    a, b = [], []
    ser(root, a)
    ser(sub, b)
    big, small = ",".join(a), ",".join(b)
    return small in big
```

**Cost:** O(m + n) time, O(m + n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every node of root, check whether the tree starting there equals subRoot. That costs O(m · n) time and O(h) space."

"I can do better: Serialise both trees in preorder with markers for None and delimiters around values; subRoot is a subtree exactly when its string is a substring. (Use KMP for a guaranteed linear search.) That's O(m + n) time and O(m + n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/subtree-of-another-tree)

<a id="q9"></a>
### Q9. Binary Tree Level Order Traversal

🟡 Interview standard · *BFS by level*

Return the node values level by level, left to right.

**Examples**

- `root = [3, 9, 20, None, None, 15, 7]` → `[[3], [9, 20], [15, 7]]`
- `root = [1]` → `[[1]]`
- `root = []` → `[]`

In [ ]:
def level_order(root):
    """
    Return the node values level by level, left to right.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(level_order, [(([3, 9, 20, None, None, 15, 7],), [[3], [9, 20], [15, 7]]), (([1],), [[1]]), (([],), [])], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Find the height, then for each depth d run a DFS that collects nodes at depth d.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

BFS with a queue; process exactly len(queue) nodes per round — that's one level.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Find the height, then for each depth d run a DFS that collects nodes at depth d.

```python
def level_order(root):
    def height(n):
        return 0 if not n else 1 + max(height(n.left), height(n.right))

    def collect(n, d, out):
        if not n:
            return
        if d == 0:
            out.append(n.val)
        collect(n.left, d - 1, out)
        collect(n.right, d - 1, out)
    res = []
    for d in range(height(root)):
        row = []
        collect(root, d, row)
        res.append(row)
    return res
```

**Cost:** O(n · h) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** BFS with a queue; process exactly len(queue) nodes per round — that's one level.

```python
def level_order(root):
    from collections import deque
    if not root:
        return []
    res, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        res.append(level)
    return res
```

**Cost:** O(n) time, O(w) — widest level space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Find the height, then for each depth d run a DFS that collects nodes at depth d. That costs O(n · h) time and O(h) space."

"I can do better: BFS with a queue; process exactly len(queue) nodes per round — that's one level. That's O(n) time and O(w) — widest level space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [3, 9, 20, None, None, 15, 7]`

1. Level 0: [3]
2. Level 1: [9, 20]
3. Level 2: [15, 7]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/binary-tree-level-order-traversal)

<a id="q10"></a>
### Q10. Lowest Common Ancestor of a Binary Search Tree

🟡 Interview standard · *BST split point*

Return the lowest node in the BST that has both `p` and `q` as descendants (a node counts as its own descendant).

**Examples**

- `root = [6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p = 2, q = 8` → `6`
- `root = [6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p = 2, q = 4` → `2`
- `root = [2, 1], p = 2, q = 1` → `2`

In [ ]:
def lca_bst(root, p, q):
    """
    Return the lowest node in the BST that has both p and q as descendants (a
    node counts as its own descendant).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lca_bst, [(([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], 2, 8), 6), (([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], 2, 4), 2), (([2, 1], 2, 1), 2)], prep=lambda a: _lca_prep(a), post=lambda n: n.val)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Use the general binary-tree LCA that ignores the ordering and searches both sides.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk from the root: both values smaller → go left; both larger → go right; otherwise this node is where they split — the LCA.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Use the general binary-tree LCA that ignores the ordering and searches both sides.

```python
def lca_bst(root, p, q):
    def lca(n):
        if not n or n.val in (p.val, q.val):
            return n
        l, r = lca(n.left), lca(n.right)
        return n if l and r else l or r
    return lca(root)
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Walk from the root: both values smaller → go left; both larger → go right; otherwise this node is where they split — the LCA.

```python
def lca_bst(root, p, q):
    node = root
    while node:
        if p.val < node.val and q.val < node.val:
            node = node.left
        elif p.val > node.val and q.val > node.val:
            node = node.right
        else:
            return node
```

**Cost:** O(h) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Use the general binary-tree LCA that ignores the ordering and searches both sides. That costs O(n) time and O(h) space."

"I can do better: Walk from the root: both values smaller → go left; both larger → go right; otherwise this node is where they split — the LCA. That's O(h) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p = 3, q = 5`

1. At 6: p = 3, q = 5
2. At 2: p = 3, q = 5
3. At 4: p = 3, q = 5

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/lowest-common-ancestor-of-a-binary-search-tree)

<a id="q11"></a>
### Q11. Lowest Common Ancestor of a Binary Tree

🟡 Interview standard · *Postorder report-up*

Return the lowest common ancestor of nodes `p` and `q` in a binary tree (no ordering).

**Examples**

- `root = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p = 5, q = 1` → `3`
- `root = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p = 5, q = 4` → `5`
- `root = [1, 2], p = 1, q = 2` → `1`

In [ ]:
def lowest_common_ancestor(root, p, q):
    """
    Return the lowest common ancestor of nodes p and q in a binary tree (no
    ordering).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lowest_common_ancestor, [(([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], 5, 1), 3), (([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], 5, 4), 5), (([1, 2], 1, 2), 1)], prep=lambda a: _lca_prep(a), post=lambda n: n.val)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Record the root-to-p and root-to-q paths, then take the last node they share.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Each call returns p or q if it finds one below it. The first node that gets a non-empty answer from both sides is the LCA.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Record the root-to-p and root-to-q paths, then take the last node they share.

```python
def lowest_common_ancestor(root, p, q):
    def path(n, target, acc):
        if not n:
            return False
        acc.append(n)
        if n is target or path(n.left, target, acc) or path(n.right, target, acc):
            return True
        acc.pop()
        return False
    a, b = [], []
    path(root, p, a)
    path(root, q, b)
    i = 0
    while i < min(len(a), len(b)) and a[i] is b[i]:
        i += 1
    return a[i - 1]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Each call returns p or q if it finds one below it. The first node that gets a non-empty answer from both sides is the LCA.

```python
def lowest_common_ancestor(root, p, q):
    top = root

    def lca(node):
        if not node or node is p or node is q:
            return node
        left, right = lca(node.left), lca(node.right)
        if left and right:
            return node
        return left or right
    return lca(root)
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Record the root-to-p and root-to-q paths, then take the last node they share. That costs O(n) time and O(n) space."

"I can do better: Each call returns p or q if it finds one below it. The first node that gets a non-empty answer from both sides is the LCA. That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4], p = 7, q = 8`

1. At 6: left reports None, right reports None
2. At 4: left reports None, right reports None
3. At 2: left reports 7, right reports None
4. At 5: left reports None, right reports 7
5. At 0: left reports None, right reports None
6. At 1: left reports None, right reports 8
7. At 3: left reports 7, right reports 8

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/lowest-common-ancestor-of-a-binary-tree)

<a id="q12"></a>
### Q12. Validate Binary Search Tree

🟡 Interview standard · *Bounds passed down*

Return `True` if the tree is a valid BST (strictly smaller on the left, strictly larger on the right, all the way down).

**Examples**

- `root = [2, 1, 3]` → `True`
- `root = [5, 1, 4, None, None, 3, 6]` → `False`
- `root = [5, 4, 6, None, None, 3, 7]` → `False`

In [ ]:
def is_valid_bst(root):
    """
    Return True if the tree is a valid BST (strictly smaller on the left,
    strictly larger on the right, all the way down).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_valid_bst, [(([2, 1, 3],), True), (([5, 1, 4, None, None, 3, 6],), False), (([5, 4, 6, None, None, 3, 7],), False)], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every node, check that all values in its left subtree are smaller and all in its right are larger.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Pass an allowed (low, high) range down: going left tightens high to the node's value, going right tightens low.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every node, check that all values in its left subtree are smaller and all in its right are larger.

```python
def is_valid_bst(root):
    def vals(n):
        return [] if not n else vals(n.left) + [n.val] + vals(n.right)
    if not root:
        return True
    if any(v >= root.val for v in vals(root.left)) or any(v <= root.val for v in vals(root.right)):
        return False
    return is_valid_bst(root.left) and is_valid_bst(root.right)
```

**Cost:** O(n²) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Pass an allowed (low, high) range down: going left tightens high to the node's value, going right tightens low.

```python
def is_valid_bst(root):
    def ok(node, low, high):
        if not node:
            return True
        good = low < node.val < high
        if not good:
            return False
        return ok(node.left, low, node.val) and ok(node.right, node.val, high)
    return ok(root, float("-inf"), float("inf"))
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every node, check that all values in its left subtree are smaller and all in its right are larger. That costs O(n²) time and O(h) space."

"I can do better: Pass an allowed (low, high) range down: going left tightens high to the node's value, going right tightens low. That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [2, 1, 3]`

1. 2 must be in (-inf, inf) → OK
2. 1 must be in (-inf, 2) → OK
3. 3 must be in (2, inf) → OK

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/validate-binary-search-tree)

<a id="q13"></a>
### Q13. Kth Smallest Element in a BST

🟡 Interview standard · *Inorder with early stop*

Return the k-th smallest value (1-indexed) in a BST.

**Examples**

- `root = [3, 1, 4, None, 2], k = 1` → `1`
- `root = [5, 3, 6, 2, 4, None, None, 1], k = 3` → `3`

In [ ]:
def kth_smallest(root, k):
    """
    Return the k-th smallest value (1-indexed) in a BST.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(kth_smallest, [(([3, 1, 4, None, 2], 1), 1), (([5, 3, 6, 2, 4, None, None, 1], 3), 3)], prep=lambda a: (mk_tree(a[0]), a[1]))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect the full inorder list and index it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Iterative inorder visits values in sorted order; stop at the k-th pop.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Collect the full inorder list and index it.

```python
def kth_smallest(root, k):
    def ino(n):
        return [] if not n else ino(n.left) + [n.val] + ino(n.right)
    return ino(root)[k - 1]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Iterative inorder visits values in sorted order; stop at the k-th pop.

```python
def kth_smallest(root, k):
    stack, cur, count = [], root, 0
    while True:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        count += 1
        if count == k:
            return cur.val
        cur = cur.right
```

**Cost:** O(h + k) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect the full inorder list and index it. That costs O(n) time and O(n) space."

"I can do better: Iterative inorder visits values in sorted order; stop at the k-th pop. That's O(h + k) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [5, 3, 6, 2, 4, None, None, 1], k = 4`

1. Inorder visit #1: 1
2. Inorder visit #2: 2
3. Inorder visit #3: 3
4. Inorder visit #4: 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/kth-smallest-element-in-a-bst)

<a id="q14"></a>
### Q14. Binary Tree Right Side View

🟡 Interview standard · *DFS right-first by depth*

Return the values visible from the right side, top to bottom.

**Examples**

- `root = [1, 2, 3, None, 5, None, 4]` → `[1, 3, 4]`
- `root = [1, 2, 3, 4, None, None, None, 5]` → `[1, 3, 4, 5]`
- `root = []` → `[]`

In [ ]:
def right_side_view(root):
    """
    Return the values visible from the right side, top to bottom.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(right_side_view, [(([1, 2, 3, None, 5, None, 4],), [1, 3, 4]), (([1, 2, 3, 4, None, None, None, 5],), [1, 3, 4, 5]), (([],), [])], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Level-order traversal storing every level, then take each level's last value.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

DFS visiting right before left; the first node reached at each new depth is the visible one.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Level-order traversal storing every level, then take each level's last value.

```python
def right_side_view(root):
    levels, q = [], [root] if root else []
    while q:
        levels.append([n.val for n in q])
        q = [c for n in q for c in (n.left, n.right) if c]
    return [lv[-1] for lv in levels]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** DFS visiting right before left; the first node reached at each new depth is the visible one.

```python
def right_side_view(root):
    res = []

    def dfs(node, depth):
        if not node:
            return
        if depth == len(res):
            res.append(node.val)
        dfs(node.right, depth + 1)
        dfs(node.left, depth + 1)

    dfs(root, 0)
    return res
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Level-order traversal storing every level, then take each level's last value. That costs O(n) time and O(n) space."

"I can do better: DFS visiting right before left; the first node reached at each new depth is the visible one. That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [1, 2, 3, None, 5, None, 4]`

1. First node at depth 0 (coming from the right): 1
2. First node at depth 1 (coming from the right): 3
3. First node at depth 2 (coming from the right): 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/binary-tree-right-side-view)

<a id="q15"></a>
### Q15. Construct Binary Tree from Preorder and Inorder Traversal

🟡 Interview standard · *Split by root index*

Build the tree from its preorder and inorder traversals (values are unique).

**Examples**

- `preorder = [3, 9, 20, 15, 7], inorder = [9, 3, 15, 20, 7]` → `[3, 9, 20, None, None, 15, 7]`
- `preorder = [-1], inorder = [-1]` → `[-1]`

In [ ]:
def build_tree(preorder, inorder):
    """
    Build the tree from its preorder and inorder traversals (values are unique).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(build_tree, [(([3, 9, 20, 15, 7], [9, 3, 15, 20, 7]), [3, 9, 20, None, None, 15, 7]), (([-1], [-1]), [-1])], post=tree_arr)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Take preorder[0] as the root, search for it in inorder to split, and slice lists for each side.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same split, but look up the root's inorder index in a dict and recurse on index ranges instead of slices.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Take preorder[0] as the root, search for it in inorder to split, and slice lists for each side.

```python
def build_tree(preorder, inorder):
    if not preorder:
        return None
    root = TreeNode(preorder[0])
    i = inorder.index(preorder[0])
    root.left = build_tree(preorder[1:i + 1], inorder[:i])
    root.right = build_tree(preorder[i + 1:], inorder[i + 1:])
    return root
```

**Cost:** O(n²) time, O(n²) slices space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Same split, but look up the root's inorder index in a dict and recurse on index ranges instead of slices.

```python
def build_tree(preorder, inorder):
    where = {v: i for i, v in enumerate(inorder)}
    pre = iter(preorder)
    holder = {}

    def build(lo, hi):
        if lo > hi:
            return None
        val = next(pre)
        node = TreeNode(val)
        holder.setdefault("root", node)
        node.left = build(lo, where[val] - 1)
        node.right = build(where[val] + 1, hi)
        return node

    return build(0, len(inorder) - 1)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Take preorder[0] as the root, search for it in inorder to split, and slice lists for each side. That costs O(n²) time and O(n²) slices space."

"I can do better: Same split, but look up the root's inorder index in a dict and recurse on index ranges instead of slices. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `preorder = [3, 9, 20, 15, 7], inorder = [9, 3, 15, 20, 7]`

1. Root 3 splits inorder[0..4] at index 1
2. Root 9 splits inorder[0..0] at index 0
3. Root 20 splits inorder[2..4] at index 3
4. Root 15 splits inorder[2..2] at index 2
5. Root 7 splits inorder[4..4] at index 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/construct-binary-tree-from-preorder-and-inorder-traversal)

<a id="q16"></a>
### Q16. Binary Tree Zigzag Level Order Traversal

🟡 Interview standard · *BFS + direction flag*

Return the level order traversal, alternating left→right and right→left.

**Examples**

- `root = [3, 9, 20, None, None, 15, 7]` → `[[3], [20, 9], [15, 7]]`
- `root = [1]` → `[[1]]`
- `root = []` → `[]`

In [ ]:
def zigzag_level_order(root):
    """
    Return the level order traversal, alternating left→right and right→left.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(zigzag_level_order, [(([3, 9, 20, None, None, 15, 7],), [[3], [20, 9], [15, 7]]), (([1],), [[1]]), (([],), [])], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Plain level order, then reverse every odd level afterwards.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

BFS per level, building each level in a deque: append right on even levels, append left on odd levels.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Plain level order, then reverse every odd level afterwards.

```python
def zigzag_level_order(root):
    levels, q = [], [root] if root else []
    while q:
        levels.append([n.val for n in q])
        q = [c for n in q for c in (n.left, n.right) if c]
    return [lv if i % 2 == 0 else lv[::-1] for i, lv in enumerate(levels)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** BFS per level, building each level in a deque: append right on even levels, append left on odd levels.

```python
def zigzag_level_order(root):
    from collections import deque
    if not root:
        return []
    res, q, ltr = [], deque([root]), True
    while q:
        level = deque()
        for _ in range(len(q)):
            node = q.popleft()
            if ltr:
                level.append(node.val)
            else:
                level.appendleft(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        res.append(list(level))
        ltr = not ltr
    return res
```

**Cost:** O(n) time, O(w) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Plain level order, then reverse every odd level afterwards. That costs O(n) time and O(n) space."

"I can do better: BFS per level, building each level in a deque: append right on even levels, append left on odd levels. That's O(n) time and O(w) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [1, 2, 3, 4, 5, 6, 7]`

1. Level 0 read left → right: [1]
2. Level 1 read right → left: [3, 2]
3. Level 2 read left → right: [4, 5, 6, 7]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/binary-tree-zigzag-level-order-traversal)

<a id="q17"></a>
### Q17. Count Good Nodes in Binary Tree

🟡 Interview standard · *Carry the max downward*

A node is good if no node on the path from the root to it has a greater value. Count good nodes.

**Examples**

- `root = [3, 1, 4, 3, None, 1, 5]` → `4`
- `root = [3, 3, None, 4, 2]` → `3`
- `root = [1]` → `1`

In [ ]:
def good_nodes(root):
    """
    A node is good if no node on the path from the root to it has a greater
    value. Count good nodes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(good_nodes, [(([3, 1, 4, 3, None, 1, 5],), 4), (([3, 3, None, 4, 2],), 3), (([1],), 1)], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each node, find its root path again and check the maximum on it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

DFS passing the largest value seen so far on the path; a node is good if its value ≥ that max.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each node, find its root path again and check the maximum on it.

```python
def good_nodes(root):
    count = 0

    def visit(n, path):
        nonlocal count
        if not n:
            return
        if all(v <= n.val for v in path):
            count += 1
        visit(n.left, path + [n.val])
        visit(n.right, path + [n.val])
    visit(root, [])
    return count
```

**Cost:** O(n · h) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** DFS passing the largest value seen so far on the path; a node is good if its value ≥ that max.

```python
def good_nodes(root):
    good = []

    def dfs(node, high):
        if not node:
            return 0
        is_good = node.val >= high
        if is_good:
            good.append(node)
        high = max(high, node.val)
        return is_good + dfs(node.left, high) + dfs(node.right, high)

    return dfs(root, root.val)
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each node, find its root path again and check the maximum on it. That costs O(n · h) time and O(h) space."

"I can do better: DFS passing the largest value seen so far on the path; a node is good if its value ≥ that max. That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [3, 1, 4, 3, None, 1, 5]`

1. 3 vs path max 3 → good
2. 1 vs path max 3 → not good
3. 3 vs path max 3 → good
4. 4 vs path max 3 → good
5. 1 vs path max 4 → not good
6. 5 vs path max 4 → good

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/count-good-nodes-in-binary-tree)

<a id="q18"></a>
### Q18. Flatten Binary Tree to Linked List

🟡 Interview standard · *Rewire in place (Morris style)*

Flatten the tree in place into a 'linked list' along right pointers in preorder; all left pointers become None.

**Examples**

- `root = [1, 2, 5, 3, 4, None, 6]` → `[1, None, 2, None, 3, None, 4, None, 5, None, 6]`
- `root = []` → `[]`
- `root = [0]` → `[0]`

In [ ]:
def flatten(root):
    """
    Flatten the tree in place into a 'linked list' along right pointers in
    preorder; all left pointers become None.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(flatten, [(([1, 2, 5, 3, 4, None, 6],), [1, None, 2, None, 3, None, 4, None, 5, None, 6]), (([],), []), (([0],), [0])], prep=T1, post=tree_arr, inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Record the preorder nodes in a list, then relink them one after another.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

For each node with a left child: find the rightmost node of the left subtree, hang the right subtree there, move the left subtree to the right. Then step right.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Record the preorder nodes in a list, then relink them one after another.

```python
def flatten(root):
    order = []

    def pre(n):
        if n:
            order.append(n)
            pre(n.left)
            pre(n.right)
    pre(root)
    for a, b in zip(order, order[1:]):
        a.left, a.right = None, b
    if order:
        order[-1].left = order[-1].right = None
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** For each node with a left child: find the rightmost node of the left subtree, hang the right subtree there, move the left subtree to the right. Then step right.

```python
def flatten(root):
    cur = root
    while cur:
        if cur.left:
            tail = cur.left
            while tail.right:
                tail = tail.right
            tail.right = cur.right
            cur.right, cur.left = cur.left, None
        cur = cur.right
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Record the preorder nodes in a list, then relink them one after another. That costs O(n) time and O(n) space."

"I can do better: For each node with a left child: find the rightmost node of the left subtree, hang the right subtree there, move the left subtree to the right. Then step right. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/flatten-binary-tree-to-linked-list)

<a id="q19"></a>
### Q19. Binary Tree Maximum Path Sum

🔴 Stretch · *Gain returned up, best through node*

A path goes along parent–child edges and need not pass through the root. Return the largest sum of node values on any non-empty path.

**Examples**

- `root = [1, 2, 3]` → `6`
- `root = [-10, 9, 20, None, None, 15, 7]` → `42`
- `root = [-3]` → `-3`

In [ ]:
def max_path_sum(root):
    """
    A path goes along parent–child edges and need not pass through the root.
    Return the largest sum of node values on any non-empty path.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_path_sum, [(([1, 2, 3],), 6), (([-10, 9, 20, None, None, 15, 7],), 42), (([-3],), -3)], prep=T1)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Treat every node as the top of the path and compute the best downward chains from it with separate calls.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Each call returns the best chain going down from the node (dropping negative branches). The path through the node is val + left gain + right gain; keep the max.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Treat every node as the top of the path and compute the best downward chains from it with separate calls.

```python
def max_path_sum(root):
    def down(n):
        return 0 if not n else max(0, n.val + max(down(n.left), down(n.right)))

    def best(n):
        if not n:
            return float("-inf")
        through = n.val + down(n.left) + down(n.right)
        return max(through, best(n.left), best(n.right))
    return best(root)
```

**Cost:** O(n²) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Each call returns the best chain going down from the node (dropping negative branches). The path through the node is val + left gain + right gain; keep the max.

```python
def max_path_sum(root):
    best = float("-inf")

    def gain(node):
        nonlocal best
        if not node:
            return 0
        l = max(gain(node.left), 0)
        r = max(gain(node.right), 0)
        best = max(best, node.val + l + r)
        return node.val + max(l, r)

    gain(root)
    return best
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Treat every node as the top of the path and compute the best downward chains from it with separate calls. That costs O(n²) time and O(h) space."

"I can do better: Each call returns the best chain going down from the node (dropping negative branches). The path through the node is val + left gain + right gain; keep the max. That's O(n) time and O(h) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [1, 2, 3]`

1. Path through 2: 2 + 0 + 0 = 2
2. Path through 3: 3 + 0 + 0 = 3
3. Path through 1: 1 + 2 + 3 = 6

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/binary-tree-maximum-path-sum)

<a id="q20"></a>
### Q20. Serialize and Deserialize Binary Tree

🔴 Stretch · *Preorder with null markers*

Design `serialize(root) → str` and `deserialize(str) → root` so that the tree round-trips exactly.

**Examples**

- `root = [1, 2, 3, None, None, 4, 5]` → `[1, 2, 3, None, None, 4, 5]`
- `root = []` → `[]`
- `root = [1, None, 2, None, 3]` → `[1, None, 2, None, 3]`

In [ ]:
def Codec():
    """
    Design serialize(root) → str and deserialize(str) → root so that the tree
    round-trips exactly.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(Codec, [(([1, 2, 3, None, None, 4, 5],), [1, 2, 3, None, None, 4, 5]), (([],), []), (([1, None, 2, None, 3],), [1, None, 2, None, 3])], runner=lambda cls, a: tree_arr(cls().deserialize(cls().serialize(mk_tree(a[0])))))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Store the tree as an array using heap positions (children of i at 2i+1, 2i+2). A skewed tree of n nodes needs about 2ⁿ slots.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Preorder walk writing '#' for every missing child. Reading tokens back in the same order rebuilds the tree, since each '#' tells us where a subtree ends.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Store the tree as an array using heap positions (children of i at 2i+1, 2i+2). A skewed tree of n nodes needs about 2ⁿ slots.

```python
class Codec:
    def serialize(self, root):
        slots = {}

        def put(n, i):
            if n:
                slots[i] = n.val
                put(n.left, 2 * i + 1)
                put(n.right, 2 * i + 2)
        put(root, 0)
        size = max(slots) + 1 if slots else 0
        return ",".join(str(slots[i]) if i in slots else "#" for i in range(size))

    def deserialize(self, data):
        if not data:
            return None
        vals = data.split(",")

        def get(i):
            if i >= len(vals) or vals[i] == "#":
                return None
            return TreeNode(int(vals[i]), get(2 * i + 1), get(2 * i + 2))
        return get(0)
```

**Cost:** O(2ⁿ) worst time, O(2ⁿ) worst space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Preorder walk writing '#' for every missing child. Reading tokens back in the same order rebuilds the tree, since each '#' tells us where a subtree ends.

```python
class Codec:
    def serialize(self, root):
        out = []

        def dfs(node):
            if not node:
                out.append("#")
                return
            out.append(str(node.val))
            dfs(node.left)
            dfs(node.right)
        dfs(root)
        return ",".join(out)

    def deserialize(self, data):
        tokens = iter(data.split(","))

        def build():
            t = next(tokens)
            if t == "#":
                return None
            node = TreeNode(int(t))
            node.left = build()
            node.right = build()
            return node
        root = build()
        return root
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Store the tree as an array using heap positions (children of i at 2i+1, 2i+2). A skewed tree of n nodes needs about 2ⁿ slots. That costs O(2ⁿ) worst time and O(2ⁿ) worst space."

"I can do better: Preorder walk writing '#' for every missing child. Reading tokens back in the same order rebuilds the tree, since each '#' tells us where a subtree ends. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `root = [1, 2, 3, None, None, 4, 5]`

1. Write 1
2. Write 2
3. Write 3
4. Write 4
5. Write 5
6. Rebuilt by reading the tokens in the same preorder

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/trees/serialize-and-deserialize-binary-tree)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What's the difference between a binary tree and a BST?</b></summary>

A binary tree only limits each node to two children. A BST adds the ordering rule: left subtree keys &lt; node &lt; right subtree keys, at every node. That rule enables O(h) search.

</details>

<details>
<summary><b>2. Depth vs height?</b></summary>

Depth of a node is measured from the root down to the node; height is measured from the node down to its deepest leaf. The root has depth 0; leaves have height 0. The tree's height is the root's height.

</details>

<details>
<summary><b>3. Complete vs full vs perfect binary tree?</b></summary>

Full: every node has 0 or 2 children. Complete: all levels filled except possibly the last, which is filled left to right. Perfect: full and all leaves at the same depth (2ʰ⁺¹ − 1 nodes).

</details>

<details>
<summary><b>4. Why does inorder traversal of a BST give sorted output?</b></summary>

Inorder visits the left subtree (all smaller keys), then the node, then the right subtree (all larger keys). Applied recursively, every key is output after all smaller keys and before all larger ones.

</details>

<details>
<summary><b>5. DFS vs BFS on trees — when to use which?</b></summary>

DFS (recursion) for problems about paths, subtrees and anything computed from children (height, diameter). BFS (queue) for level-based questions (level order, right view, minimum depth). Memory: DFS O(h), BFS O(w).

</details>

<details>
<summary><b>6. What is the worst-case time of BST search and how is it avoided?</b></summary>

O(n) when the tree is skewed (e.g., inserting sorted data). Self-balancing trees — AVL or Red–Black — rotate nodes after updates to keep the height O(log n).

</details>

<details>
<summary><b>7. How do you delete a node with two children from a BST?</b></summary>

Replace its key with its inorder successor (the minimum of its right subtree) — or predecessor — then delete that successor node, which has at most one child.

</details>

<details>
<summary><b>8. Can a tree be rebuilt from its preorder and inorder traversals?</b></summary>

Yes, if keys are distinct: preorder's first element is the root; its position in inorder splits the left and right subtrees; recurse. Preorder + postorder alone is ambiguous when nodes have one child.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Tree vocabulary
- [ ] I can explain: Recursion on trees
- [ ] I can explain: Breadth-first search: level by level
- [ ] I can explain: Bottom-up: return a value, update a global
- [ ] I can explain: Top-down: pass information down
- [ ] I can explain: Binary search trees
- [ ] I can explain: Building, serialising and ancestors

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Binary Tree Zigzag Level Order Traversal
- [ ] [Q17](#q17) Count Good Nodes in Binary Tree
- [ ] [Q18](#q18) Flatten Binary Tree to Linked List
- [ ] [Q19](#q19) Binary Tree Maximum Path Sum
- [ ] [Q20](#q20) Serialize and Deserialize Binary Tree

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.